# Phase 2 — YOLO26M detection on one video

**REQUIREMENT:** prove full-frame detection on one actual video in Kaggle, using one explicitly selected GPU. This notebook does not implement tracking, ReID, geometry, crossing, counting, stabilization, enhancement, UI, or GPU scheduling.

**FACT:** `TOR3-PAGI.mp4` was discovered and visually inspected in Phase 1. The official `yolo26m.pt` asset and Ultralytics 8.4.175 package/API were verified against published sources; this does not establish Kaggle compatibility or detector quality before execution.

**PROPOSAL — initial experiment:** retain the pinned package's published prediction defaults unless an explicit override is configured. These are baseline settings, not selected product thresholds. Retain every model class. `bus` does not establish an `angkot` class or accepted reporting-category mapping.

Restart the kernel and Run All with Kaggle GPU and Internet enabled for each new experiment. Reusing the inference cell can reuse predictor warmup/state and old samples. Raw videos, weights, bulk artifacts, and executed notebook outputs stay outside GitHub; curated original/annotated samples and reports may be archived as evidence. There is no ground truth; this is an executable detection experiment, not an accuracy benchmark.

Sources: [model](https://docs.ultralytics.com/models/yolo26/), [predict](https://docs.ultralytics.com/modes/predict/), [pinned source](https://github.com/ultralytics/ultralytics/tree/v8.4.175), [official asset release](https://github.com/ultralytics/assets/releases/tag/v8.4.0).


In [ ]:
# Explicit experiment configuration. Change the expected hash too when changing video.
from pathlib import Path

DATASET_ROOT = Path('/kaggle/input/datasets/chrisbiran/traffic-tracker-videos')
VIDEO_PATH = DATASET_ROOT / 'TOR3-PAGI.mp4'
EXPECTED_VIDEO_SHA256 = '606cfdd26c2d0a6892f5680d6a35d7e4d372efd76dd6bc56e2e4cbed7a74754c'
OUTPUT_ROOT = Path('/kaggle/working/yolo_detection')
DEVICE = 0                       # One GPU; no automatic CPU fallback or orchestration.
FRAME_LIMIT = None               # None: sequentially process until decoder read=False.
SEED = 0                        # PROPOSAL: record seeds, without promising deterministic CUDA.
INSTALL_DEPENDENCIES = True
ULTRALYTICS_VERSION = '8.4.175'
WEIGHTS_URL = 'https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26m.pt'
WEIGHTS_SHA256 = '401cea9ab23ad19246ff7744859816bc599f350e93c9dd30367b6f0a0745d0b7'
WEIGHTS_BYTES = 44255705

# None means do not override the pinned package's published setting.
CONF = None
IMGSZ = None
IOU = None
MAX_DET = None
NMS = None                     # Source default: auto / one-to-many + external NMS.
QUANTIZE = None                # Source default: FP32; no guessed torch/CUDA pin.
MANUAL_DETECTION_REVIEW = {
    'visible_detections': {'observation': None, 'evidence': []},
    'missed_or_ambiguous_objects': {'observation': None, 'evidence': []},
    'class_ambiguities': {'observation': None, 'evidence': []},
    'overlay_and_occlusion_cases': {'observation': None, 'evidence': []},
    'questions_for_later_experiments': {'observation': None, 'evidence': [], 'claim_type': 'OPEN'},
}


In [ ]:
# Create run-local provenance before installing/loading anything.
import contextlib
import csv
import datetime as dt
import hashlib
import importlib.metadata
import io
import json
import math
import os
import platform
import random
import shutil
import statistics
import subprocess
import sys
import time
import traceback
import urllib.request
import uuid

RUN_ID = dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ_') + uuid.uuid4().hex[:8]
RUN_DIR = OUTPUT_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)


def clean_json(value):
    if isinstance(value, dict):
        return {str(key): clean_json(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean_json(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    return str(value)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


MANIFEST = {
    'run_id': RUN_ID, 'started_utc': dt.datetime.now(dt.timezone.utc).isoformat(),
    'status': 'created', 'phase': 2, 'claim_type': 'EXPERIMENT_RESULT_ON_EXECUTION',
    'scope': 'one video, full-frame detection only, no persistent identities or traffic counts',
    'configuration': {
        'video_path': str(VIDEO_PATH), 'expected_video_sha256': EXPECTED_VIDEO_SHA256,
        'device': DEVICE, 'frame_limit': FRAME_LIMIT, 'seed': SEED,
        'package_version': ULTRALYTICS_VERSION,
        'baseline_status': 'PROPOSAL: pinned published defaults; not tuned product settings',
        'overrides': {'conf': CONF, 'imgsz': IMGSZ, 'iou': IOU, 'max_det': MAX_DET,
                      'nms': NMS, 'quantize': QUANTIZE},
        'class_filter': None, 'detection_roi': None, 'enhancement': None,
    },
    'weights': {'url': WEIGHTS_URL, 'expected_sha256': WEIGHTS_SHA256, 'expected_bytes': WEIGHTS_BYTES},
    'input': None, 'environment': None, 'model': None, 'effective_predictor_arguments': None,
    'execution': {'frames_processed': 0, 'decode_stop': None},
    'artifacts': {}, 'sample_frames': [],
    'limitations': [
        'No ground truth; detector scores and plausible boxes do not establish accuracy.',
        'Detection records describe individual frames, not unique vehicles or traffic counts.',
        'CAP_PROP_POS_MSEC is backend-reported, not verified source presentation timestamps.',
        'Frame-index / FPS estimates are separate from backend timestamps and may disagree.',
        'OpenCV read=False alone cannot distinguish genuine end-of-file from decode failure.',
        'Matching a metadata frame total is a consistency check, not proof of source integrity.',
        'No bus/angkot reporting-category mapping is accepted by this experiment.',
        'One morning video does not establish night or multi-video performance.',
    ],
}


def save_report():
    MANIFEST['manual_detection_review'] = clean_json(MANUAL_DETECTION_REVIEW)
    MANIFEST['manual_review_status'] = ('recorded_not_a_benchmark' if any(
        item.get('observation') is not None for item in MANUAL_DETECTION_REVIEW.values()
    ) else 'pending_actual_annotated_frame_review')
    serialized = json.dumps(clean_json(MANIFEST), ensure_ascii=False, indent=2, allow_nan=False)
    (RUN_DIR / 'run_manifest.json').write_text(serialized + '\n', encoding='utf-8')
    (RUN_DIR / 'detection_report.json').write_text(serialized + '\n', encoding='utf-8')
    sections = [
        '# Phase 2 detection report',
        f"Run: `{RUN_ID}`; status: `{MANIFEST['status']}`.",
        'This report contains executed observations only when a run has actually completed.',
    ]
    for title, field in [('Input identity / metadata', 'input'), ('Environment', 'environment'),
                         ('Actual model / class labels', 'model'), ('Configuration', 'configuration'),
                         ('Effective predictor arguments', 'effective_predictor_arguments'),
                         ('Execution / measured timings', 'execution'), ('Artifacts', 'artifacts'),
                         ('Manual detection review', 'manual_detection_review')]:
        sections += [f'## {title}', '```json\n' + json.dumps(clean_json(MANIFEST.get(field)),
                     ensure_ascii=False, indent=2, allow_nan=False) + '\n```']
    sections += ['## Limitations', '\n'.join('- ' + item for item in MANIFEST['limitations'])]
    if MANIFEST.get('error'):
        sections += ['## Execution failure', '```json\n' + json.dumps(MANIFEST['error'],
                     ensure_ascii=False, indent=2) + '\n```']
    (RUN_DIR / 'detection_report.md').write_text('\n\n'.join(sections) + '\n', encoding='utf-8')


@contextlib.contextmanager
def stage(name):
    MANIFEST['current_stage'] = name
    save_report()
    started = time.perf_counter()
    try:
        yield
    except BaseException as exc:
        MANIFEST['status'] = 'failed'
        MANIFEST['error'] = {'stage': name, 'type': type(exc).__name__, 'message': str(exc)}
        MANIFEST['finished_utc'] = dt.datetime.now(dt.timezone.utc).isoformat()
        raise
    finally:
        MANIFEST.setdefault('stage_seconds', {})[name] = time.perf_counter() - started
        save_report()


with stage('configuration_validation'):
    if not isinstance(DEVICE, int) or isinstance(DEVICE, bool) or DEVICE < 0:
        raise ValueError('DEVICE must be one non-negative CUDA device index.')
    if FRAME_LIMIT is not None and (not isinstance(FRAME_LIMIT, int) or isinstance(FRAME_LIMIT, bool) or FRAME_LIMIT <= 0):
        raise ValueError('FRAME_LIMIT must be None or a positive integer.')
    if not isinstance(VIDEO_PATH, Path) or not VIDEO_PATH.is_file():
        raise FileNotFoundError(f'Actual video is unavailable: {VIDEO_PATH}. Execute with the dataset mounted.')
print('Run artifacts:', RUN_DIR)


## Install and verify the execution environment

The notebook pins the verified Ultralytics release. Pip resolves its declared dependencies against the existing Kaggle runtime; it does not blindly upgrade or select a PyTorch/CUDA build. Actual installed versions are retained. If dependency import or CUDA verification fails, retain the failure report and investigate rather than switching detector/device silently.


In [ ]:
with stage('dependency_installation'):
    if INSTALL_DEPENDENCIES:
        subprocess.run([sys.executable, '-m', 'pip', 'install',
                        f'ultralytics=={ULTRALYTICS_VERSION}'], check=True)

with stage('environment_verification'):
    settings_dir = OUTPUT_ROOT / '.ultralytics_settings'
    settings_dir.mkdir(parents=True, exist_ok=True)
    os.environ['YOLO_CONFIG_DIR'] = str(settings_dir)
    import cv2
    import numpy as np
    import torch
    import torchvision
    import ultralytics
    from PIL import Image, ImageDraw
    from IPython.display import display
    from ultralytics import YOLO

    if importlib.metadata.version('ultralytics') != ULTRALYTICS_VERSION:
        raise RuntimeError('Installed Ultralytics differs from the explicitly pinned version.')
    versions = {}
    for package in ('ultralytics', 'torch', 'torchvision', 'numpy', 'opencv-python', 'opencv-python-headless', 'Pillow'):
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = None
    hardware = {'python': sys.version, 'platform': platform.platform(),
                'cpu_count': os.cpu_count(), 'package_versions': versions,
                'opencv_version': cv2.__version__, 'ultralytics_settings_directory': str(settings_dir),
                'torch_cuda_build': torch.version.cuda,
                'cuda_available': torch.cuda.is_available(), 'visible_cuda_device_count': torch.cuda.device_count(),
                'cuda_devices': [], 'nvidia_smi': None}
    for index in range(torch.cuda.device_count()):
        properties = torch.cuda.get_device_properties(index)
        hardware['cuda_devices'].append({'index': index, 'name': properties.name,
            'total_memory_bytes': properties.total_memory, 'compute_capability': list(torch.cuda.get_device_capability(index))})
    if shutil.which('nvidia-smi'):
        try:
            probe = subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version',
                                    '--format=csv,noheader,nounits'], capture_output=True, text=True, timeout=15)
            hardware['nvidia_smi'] = {'returncode': probe.returncode,
                                     'stdout': probe.stdout.strip(), 'stderr': probe.stderr.strip()}
        except (OSError, subprocess.TimeoutExpired) as exc:
            hardware['nvidia_smi'] = {'error': str(exc)}
    MANIFEST['environment'] = hardware
    if not hardware['cuda_available'] or DEVICE >= hardware['visible_cuda_device_count']:
        raise RuntimeError(f'CUDA device {DEVICE} is unavailable. Enable Kaggle GPU; no CPU fallback is enabled.')
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)  # Seeding visible devices does not schedule work on them.
    hardware['reproducibility'] = {'seed': SEED,
        'deterministic_algorithms_enabled': torch.are_deterministic_algorithms_enabled(),
        'cudnn_deterministic': torch.backends.cudnn.deterministic,
        'cudnn_benchmark': torch.backends.cudnn.benchmark,
        'exact_repeatability_claimed': False}
    print(json.dumps(clean_json(hardware), ensure_ascii=False, indent=2))


In [ ]:
with stage('input_identity_and_metadata'):
    hash_started = time.perf_counter()
    video_sha256 = sha256_file(VIDEO_PATH)
    hash_seconds = time.perf_counter() - hash_started
    MANIFEST['input'] = {'filename': VIDEO_PATH.name, 'path': str(VIDEO_PATH),
                         'file_size_bytes': VIDEO_PATH.stat().st_size, 'sha256': video_sha256,
                         'hash_seconds': hash_seconds, 'expected_hash_matches': None,
                         'recording_period_assignment': 'unknown: filename is not authoritative metadata',
                         'store_location_assignment': 'unknown', 'ffprobe': None}
    if EXPECTED_VIDEO_SHA256 is not None:
        MANIFEST['input']['expected_hash_matches'] = video_sha256 == EXPECTED_VIDEO_SHA256
        if not MANIFEST['input']['expected_hash_matches']:
            raise RuntimeError('Input differs from the SHA-identified Phase 1 video. Review VIDEO_PATH and expected hash.')
    probe_cap = cv2.VideoCapture(str(VIDEO_PATH))
    try:
        if not probe_cap.isOpened():
            raise RuntimeError('OpenCV could not open the selected actual video.')
        observed = {'width': probe_cap.get(cv2.CAP_PROP_FRAME_WIDTH),
                    'height': probe_cap.get(cv2.CAP_PROP_FRAME_HEIGHT),
                    'fps': probe_cap.get(cv2.CAP_PROP_FPS),
                    'frame_count': probe_cap.get(cv2.CAP_PROP_FRAME_COUNT),
                    'fourcc_integer': probe_cap.get(cv2.CAP_PROP_FOURCC),
                    'backend': probe_cap.getBackendName()}
    finally:
        probe_cap.release()
    for key, value in list(observed.items()):
        if isinstance(value, float) and (not math.isfinite(value) or value < 0):
            observed[key] = None
    valid_count = observed['frame_count']
    METADATA_FRAME_COUNT = int(valid_count) if valid_count and float(valid_count).is_integer() else None
    valid_fps = observed['fps']
    ESTIMATED_FPS = float(valid_fps) if valid_fps and valid_fps > 0 else None
    observed['duration_seconds_estimate_from_count_fps'] = (
        METADATA_FRAME_COUNT / ESTIMATED_FPS if METADATA_FRAME_COUNT and ESTIMATED_FPS else None)
    fourcc = observed['fourcc_integer']
    observed['fourcc_text'] = ''.join(chr((int(fourcc) >> (8 * shift)) & 255) for shift in range(4)) if fourcc else None
    MANIFEST['input']['opencv_metadata'] = observed
    if shutil.which('ffprobe'):
        try:
            result = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0',
                '-show_entries', 'stream=codec_name,width,height,avg_frame_rate,r_frame_rate,nb_frames,duration:format=duration',
                '-of', 'json', str(VIDEO_PATH)], capture_output=True, text=True, timeout=30)
            MANIFEST['input']['ffprobe'] = {'returncode': result.returncode,
                'data': json.loads(result.stdout) if result.returncode == 0 else None,
                'stderr': result.stderr.strip()}
        except (OSError, subprocess.TimeoutExpired, json.JSONDecodeError) as exc:
            MANIFEST['input']['ffprobe'] = {'error': str(exc)}
    planned_frames = min(METADATA_FRAME_COUNT, FRAME_LIMIT) if METADATA_FRAME_COUNT and FRAME_LIMIT else (FRAME_LIMIT or METADATA_FRAME_COUNT)
    SAMPLE_INDICES = sorted({round((planned_frames - 1) * fraction) for fraction in (0, .25, .5, .75, 1)}) if planned_frames else [0]
    MANIFEST['configuration']['sample_indices'] = SAMPLE_INDICES
    MANIFEST['configuration']['sampling_basis'] = ('configured prefix / metadata frame count; actual last processed frame is also saved'
        if planned_frames else 'unknown frame total: first and actual last processed frames only')
    print(json.dumps(clean_json(MANIFEST['input']), ensure_ascii=False, indent=2))


In [ ]:
# Download only the verified official asset, checking bytes and SHA before loading.
with stage('weights_download_and_verification'):
    weights_dir = OUTPUT_ROOT / 'weights'
    weights_dir.mkdir(parents=True, exist_ok=True)
    WEIGHTS_PATH = weights_dir / 'yolo26m.pt'
    download_started = time.perf_counter()
    if not WEIGHTS_PATH.is_file() or sha256_file(WEIGHTS_PATH) != WEIGHTS_SHA256:
        temporary = weights_dir / ('yolo26m.' + uuid.uuid4().hex + '.download')
        try:
            request = urllib.request.Request(WEIGHTS_URL, headers={'User-Agent': 'traffic-vehicle-counter-phase2/1'})
            with urllib.request.urlopen(request, timeout=90) as response, temporary.open('wb') as output:
                while True:
                    chunk = response.read(1024 * 1024)
                    if not chunk:
                        break
                    output.write(chunk)
            if temporary.stat().st_size != WEIGHTS_BYTES or sha256_file(temporary) != WEIGHTS_SHA256:
                raise RuntimeError('Official weights failed size/SHA verification; nothing will be loaded.')
            os.replace(temporary, WEIGHTS_PATH)
        finally:
            if temporary.exists():
                temporary.unlink()
    if WEIGHTS_PATH.stat().st_size != WEIGHTS_BYTES or sha256_file(WEIGHTS_PATH) != WEIGHTS_SHA256:
        raise RuntimeError('Cached weights failed verification.')
    MANIFEST['weights'].update({'local_path': str(WEIGHTS_PATH), 'actual_sha256': sha256_file(WEIGHTS_PATH),
                               'actual_bytes': WEIGHTS_PATH.stat().st_size,
                               'cache_or_download_seconds': time.perf_counter() - download_started})

with stage('actual_model_loading'):
    model = YOLO(str(WEIGHTS_PATH))
    if model.task != 'detect':
        raise RuntimeError(f'Expected detection task, found {model.task!r}.')
    CLASS_NAMES = {int(key): str(value) for key, value in model.names.items()}
    checkpoint_head = model.model.model[-1]
    MANIFEST['model'] = {'task': model.task, 'class_names': CLASS_NAMES,
        'model_python_type': type(model.model).__module__ + '.' + type(model.model).__name__,
        'checkpoint_head_python_type': type(checkpoint_head).__module__ + '.' + type(checkpoint_head).__name__,
        'checkpoint_head_end2end': getattr(checkpoint_head, 'end2end', None),
        'checkpoint_head_reg_max': getattr(checkpoint_head, 'reg_max', None),
        'active_predictor_backend': None,
        'observed_labels_related_to_product': {str(key): value for key, value in CLASS_NAMES.items()
                                              if value in {'person', 'motorcycle', 'car', 'bus', 'truck', 'angkot'}},
        'product_category_mapping': 'OPEN; raw model classes are retained, no bus-to-angkot inference'}
    print(json.dumps(clean_json(MANIFEST['model']), ensure_ascii=False, indent=2))


## Sequential full-frame inference and evidence retention

`FRAME_LIMIT=None` attempts the whole selected video. A positive limit is explicitly reported as a bounded prefix, even if its length happens to match metadata. Every decoded frame produces one JSONL row, including frames with no detections. Bounding boxes use native decoded pixel `xyxy` coordinates; frame indices start at zero. Model scores are not correctness probabilities or accuracy metrics.

The first prediction includes predictor setup and library warmup; its external wall time is separate from steady subsequent calls. CUDA synchronization brackets prediction timing. Decoder, serialization/sample IO, model loading, and asset acquisition are separate from inference timing. No optimization or equivalent-system benchmark is claimed.

Five metadata-based representative positions and the actual last processed frame retain both original and annotated PNGs. These samples can miss difficult events; later manual inspection must cite retained images rather than fabricate a review.


In [ ]:
def save_sample(frame_index, raw_frame, result, reason):
    samples_dir = RUN_DIR / 'samples'
    samples_dir.mkdir(exist_ok=True)
    raw_path = samples_dir / f'frame_{frame_index:09d}_raw.png'
    annotated_path = samples_dir / f'frame_{frame_index:09d}_annotated.png'
    if not cv2.imwrite(str(raw_path), raw_frame):
        raise OSError(f'Unable to retain {raw_path}.')
    if not cv2.imwrite(str(annotated_path), result.plot()):
        raise OSError(f'Unable to retain {annotated_path}.')
    record = {'frame_index': frame_index, 'reason': reason,
              'raw': str(raw_path.relative_to(RUN_DIR)), 'raw_sha256': sha256_file(raw_path),
              'annotated': str(annotated_path.relative_to(RUN_DIR)), 'annotated_sha256': sha256_file(annotated_path)}
    MANIFEST['sample_frames'] = [item for item in MANIFEST['sample_frames'] if item['frame_index'] != frame_index] + [record]
    return record


def make_contact_sheet():
    records = sorted(MANIFEST['sample_frames'], key=lambda item: item['frame_index'])
    if not records:
        return None
    columns, tile_width, image_height, label_height = 3, 512, 288, 28
    rows = math.ceil(len(records) / columns)
    canvas = Image.new('RGB', (columns * tile_width, rows * (image_height + label_height)), 'white')
    draw = ImageDraw.Draw(canvas)
    for position, record in enumerate(records):
        x, y = (position % columns) * tile_width, (position // columns) * (image_height + label_height)
        with Image.open(RUN_DIR / record['annotated']) as original:
            preview = original.convert('RGB')
            preview.thumbnail((tile_width, image_height))
            canvas.paste(preview, (x + (tile_width - preview.width) // 2, y))
        draw.text((x + 5, y + image_height + 5), f"frame {record['frame_index']} / annotated", fill='black')
    contact_path = RUN_DIR / 'annotated_contact_sheet.png'
    canvas.save(contact_path)
    return {'path': contact_path.name, 'sha256': sha256_file(contact_path), 'purpose': 'display-only resized preview; native PNGs retained'}


PREDICT_KWARGS = {'device': DEVICE, 'verbose': False, 'save': False, 'classes': None}
for key, value in {'conf': CONF, 'imgsz': IMGSZ, 'iou': IOU, 'max_det': MAX_DET,
                   'nms': NMS, 'quantize': QUANTIZE}.items():
    if value is not None:
        PREDICT_KWARGS[key] = value

# A first call can include setup and warmup; only later calls enter steady prediction summaries.
with stage('sequential_detection'):
    MANIFEST['status'] = 'running'
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    frames_processed = 0
    decode_seconds = 0.0
    prediction_wall_ms = []
    speed_ms = {'preprocess': [], 'inference': [], 'postprocess': []}
    first_call_seconds = None
    last_sample = None
    stop_reason = None
    loop_started = time.perf_counter()
    torch.cuda.reset_peak_memory_stats(DEVICE)
    records_path = RUN_DIR / 'frame_detections.jsonl'
    MANIFEST['artifacts']['frame_detections'] = records_path.name
    try:
        if not cap.isOpened():
            raise RuntimeError('OpenCV could not open the video for sequential inference.')
        with records_path.open('w', encoding='utf-8') as output:
            while True:
                if FRAME_LIMIT is not None and frames_processed >= FRAME_LIMIT:
                    stop_reason = 'configured_frame_limit'
                    break
                decode_started = time.perf_counter()
                ok, frame = cap.read()
                decode_seconds += time.perf_counter() - decode_started
                if not ok:
                    stop_reason = 'opencv_read_false_eof_or_decode_failure_unknown'
                    break
                frame_index = frames_processed
                height, width = frame.shape[:2]
                reported_ms = float(cap.get(cv2.CAP_PROP_POS_MSEC))
                backend_ms = reported_ms if math.isfinite(reported_ms) and reported_ms >= 0 else None
                torch.cuda.synchronize(DEVICE)
                prediction_started = time.perf_counter()
                predictions = model.predict(source=frame, **PREDICT_KWARGS)
                torch.cuda.synchronize(DEVICE)
                prediction_seconds = time.perf_counter() - prediction_started
                if len(predictions) != 1:
                    raise RuntimeError('Expected exactly one prediction result for one decoded frame.')
                result = predictions[0]
                if tuple(result.orig_shape) != (height, width):
                    raise RuntimeError('Result coordinate space differs from original decoded frame.')
                if frame_index == 0:
                    first_call_seconds = prediction_seconds
                    MANIFEST['effective_predictor_arguments'] = clean_json(vars(model.predictor.args))
                    backend = model.predictor.model
                    MANIFEST['model']['active_predictor_backend'] = {
                        'python_type': type(backend).__module__ + '.' + type(backend).__name__,
                        'device': str(backend.device), 'format': getattr(backend, 'format', None),
                        'end2end': getattr(backend, 'end2end', None), 'fp16': getattr(backend, 'fp16', None),
                        'actual_imgsz': clean_json(model.predictor.imgsz),
                    }
                    if backend.device.type != 'cuda' or backend.device.index != DEVICE:
                        raise RuntimeError('Predictor did not use the explicitly requested CUDA device.')
                else:
                    prediction_wall_ms.append(prediction_seconds * 1000)
                    for name in speed_ms:
                        measured = result.speed.get(name)
                        if measured is not None and math.isfinite(float(measured)):
                            speed_ms[name].append(float(measured))
                detections = []
                if result.boxes is not None:
                    coordinates = result.boxes.xyxy.detach().cpu().numpy()
                    confidences = result.boxes.conf.detach().cpu().numpy()
                    classes = result.boxes.cls.detach().cpu().numpy()
                    if not (len(coordinates) == len(confidences) == len(classes)):
                        raise RuntimeError('Detector fields have inconsistent lengths.')
                    for bbox, confidence, raw_class in zip(coordinates, confidences, classes):
                        x1, y1, x2, y2 = map(float, bbox)
                        confidence, raw_class = float(confidence), float(raw_class)
                        if not all(math.isfinite(value) for value in (x1, y1, x2, y2, confidence, raw_class)):
                            raise ValueError('A detector field is not finite.')
                        class_id = int(raw_class)
                        if raw_class != class_id or class_id not in CLASS_NAMES:
                            raise ValueError('Detector class ID is outside actual model labels.')
                        if not (0 <= confidence <= 1 and 0 <= x1 <= x2 <= width and 0 <= y1 <= y2 <= height):
                            raise ValueError('Detector confidence or native-pixel bounding box is invalid.')
                        detections.append({'class_id': class_id, 'class_name': CLASS_NAMES[class_id],
                                           'confidence': confidence, 'bbox_xyxy_pixels': [x1, y1, x2, y2]})
                record = {'video_sha256': video_sha256, 'frame_index': frame_index,
                    'width': width, 'height': height, 'timestamp_backend_ms': backend_ms,
                    'timestamp_estimate_from_index_fps_seconds': frame_index / ESTIMATED_FPS if ESTIMATED_FPS else None,
                    'prediction_wall_seconds': prediction_seconds, 'first_call_includes_setup_and_warmup': frame_index == 0,
                    'library_speed_ms': clean_json(result.speed), 'detections': detections}
                output.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + '\n')
                frames_processed += 1
                MANIFEST['execution']['frames_processed'] = frames_processed
                last_sample = (frame_index, frame, result)
                if frame_index in SAMPLE_INDICES:
                    save_sample(frame_index, frame, result, 'representative_metadata_or_prefix_position')
                if frames_processed % 250 == 0:
                    print(f'{frames_processed} frames processed; elapsed {time.perf_counter() - loop_started:.1f}s')
            if frames_processed == 0:
                raise RuntimeError('No actual frames were decoded; no successful detection experiment is claimed.')
    finally:
        cap.release()
        loop_seconds = time.perf_counter() - loop_started
        if last_sample is not None and last_sample[0] not in {item['frame_index'] for item in MANIFEST['sample_frames']}:
            save_sample(*last_sample, reason='actual_last_processed_frame')
        MANIFEST['sample_frames'].sort(key=lambda item: item['frame_index'])
        MANIFEST['execution'].update({
            'frames_processed': frames_processed, 'decode_stop': stop_reason,
            'metadata_frame_count': METADATA_FRAME_COUNT,
            'processed_frames_match_metadata': frames_processed == METADATA_FRAME_COUNT if METADATA_FRAME_COUNT else None,
            'configured_frame_limit': FRAME_LIMIT,
            'scope': 'bounded_prefix' if FRAME_LIMIT is not None else 'all_frames_returned_by_decoder',
            'loop_wall_seconds': loop_seconds, 'decode_read_wall_seconds': decode_seconds,
            'first_prediction_seconds_including_setup_warmup': first_call_seconds,
            'subsequent_prediction_calls': len(prediction_wall_ms),
            'subsequent_prediction_wall_ms_mean': statistics.mean(prediction_wall_ms) if prediction_wall_ms else None,
            'subsequent_prediction_wall_ms_median': statistics.median(prediction_wall_ms) if prediction_wall_ms else None,
            'subsequent_library_speed_ms_mean': {name: statistics.mean(values) if values else None for name, values in speed_ms.items()},
            'whole_loop_throughput_frames_per_second': frames_processed / loop_seconds if loop_seconds > 0 else None,
            'cuda_peak_allocated_bytes': torch.cuda.max_memory_allocated(DEVICE),
            'cuda_peak_reserved_bytes': torch.cuda.max_memory_reserved(DEVICE),
            'timing_limits': 'Loop includes decoder/predict/JSONL/sample IO; first-call wall includes predictor setup and warmup. Library stage timings are separate.',
        })
        if records_path.is_file():
            MANIFEST['artifacts']['frame_detections_sha256'] = sha256_file(records_path)
        MANIFEST['artifacts']['annotated_contact_sheet'] = make_contact_sheet()
        save_report()
    # Independently verify the serialized stream, without loading it all into memory.
    with records_path.open(encoding='utf-8') as handle:
        verified_rows = 0
        for verified_rows, line in enumerate(handle, start=1):
            row = json.loads(line)
            if row['frame_index'] != verified_rows - 1 or row['video_sha256'] != video_sha256:
                raise ValueError('Retained detection rows are not sequential or have mixed input identity.')
        if verified_rows != frames_processed:
            raise ValueError('Retained detection row total does not match processed frames.')
    MANIFEST['execution']['jsonl_rows_verified'] = verified_rows
    MANIFEST['execution']['serialization_validation'] = 'passed'
    MANIFEST['status'] = ('completed_bounded_prefix_manual_review_pending' if FRAME_LIMIT is not None
                          else 'completed_decoder_stream_manual_review_pending')
    MANIFEST['finished_utc'] = dt.datetime.now(dt.timezone.utc).isoformat()
print(json.dumps(clean_json(MANIFEST['execution']), ensure_ascii=False, indent=2))


In [ ]:
# Inspect originals and native annotations; contact-sheet resizing is display only.
for sample in MANIFEST['sample_frames']:
    print('frame:', sample['frame_index'], '| raw:', sample['raw'], '| annotated:', sample['annotated'])
contact = MANIFEST['artifacts'].get('annotated_contact_sheet')
if contact:
    display(Image.open(RUN_DIR / contact['path']))
print('For detail, open the retained native PNGs under:', RUN_DIR / 'samples')


## Manual detection observations

Review the actual annotated samples together with their original PNGs. Edit `MANUAL_DETECTION_REVIEW` below using concrete observations and run-relative evidence paths, then run the final report cell. Leave unsupported entries `None`; a successful GPU run is not evidence that every vehicle was detected or classified correctly.

Questions may concern small distant vehicles, motorcycles/people overlap, burned-in text, cropping at frame boundaries, or confusing classes. Do not report formal accuracy or detector improvement. Night detection and enhancement need separate authorized experiments; Phase 7 owns night preprocessing.


In [ ]:
# Fill only AFTER reviewing retained actual outputs. Do not copy hypothetical observations.
# Example shape only (no result claimed):
# MANUAL_DETECTION_REVIEW['visible_detections'] = {
#     'observation': 'Concrete observation from an executed frame.',
#     'evidence': ['samples/frame_000000000_raw.png', 'samples/frame_000000000_annotated.png'],
# }
for field, item in MANUAL_DETECTION_REVIEW.items():
    if item.get('observation') is not None and not item.get('evidence'):
        raise ValueError(f'{field}: a manual observation requires retained evidence.')
    for reference in item.get('evidence', []):
        relative = Path(reference)
        if relative.is_absolute() or '..' in relative.parts or not (RUN_DIR / relative).is_file():
            raise ValueError(f'{field}: evidence must reference an existing run-relative artifact: {reference}')
        if not (RUN_DIR / relative).resolve().is_relative_to(RUN_DIR.resolve()):
            raise ValueError(f'{field}: evidence escapes this run.')
if any(item.get('observation') is not None for item in MANUAL_DETECTION_REVIEW.values()):
    if MANIFEST['status'] in {'completed_bounded_prefix_manual_review_pending', 'completed_decoder_stream_manual_review_pending'}:
        MANIFEST['status'] = MANIFEST['status'].replace('manual_review_pending', 'manual_review_recorded')
save_report()
print('Retained report:', RUN_DIR / 'detection_report.md')
print('Machine-readable report:', RUN_DIR / 'detection_report.json')
print('Phase 2 only. Phase 3 tracking has not been implemented or authorized by this notebook.')
